# Anti-Goal Chess Benchmark @ NeurIPS 2026 — CHECK MODE (tiny, raises on failure)

Paired win/lose small-model chess study (see README). Results land in `results_check/` and are zipped for download.
- Positions + exact oracles: committed (`data/positions/`), generated once by `scripts/generate_positions.py`.
- Engine + dataset tests gate every run: `scripts/test_engine.py`.
- Sweep: `scripts/run_suite.py` (models x tasks x {{win,lose}}).

## 1. Clone the repo

The repo is **private**, so this cell needs a GitHub PAT. On Kaggle, a secret only reaches the notebook if you ATTACH it: open this notebook -> right-hand **Secrets** panel -> enable `GITHUB_TOKEN` for this notebook. The token must have `repo` scope (classic PAT) or Contents:Read (fine-grained).

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "neuro-symbolic-pathfinding"
if not REPO.exists():
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""
    print("GITHUB_TOKEN found in environment:", bool(token), flush=True)
    url = "https://github.com/Vedang-P/neuro-symbolic-pathfinding.git"
    if token:
        url = url.replace("https://", f"https://x-access-token:{token}@")
    res = subprocess.run(["git", "clone", "--quiet", url, str(REPO)],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            "git clone failed. If the repo is private this means the secret never "
            "reached the notebook. Fix: notebook right-hand panel -> Secrets -> "
            "enable GITHUB_TOKEN for THIS notebook (it must contain a PAT with the "
            "repo scope). Stderr: " + res.stderr[-300:]
        )
os.chdir(REPO)
print("cwd:", Path.cwd())

## 2. Submodule + dependencies

In [ ]:
subprocess.run(["git", "submodule", "update", "--init", "--depth", "1"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "python-chess"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "wandb"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Stage runner (never raises; the verdict cell checks results)

In [ ]:
import json, time, shutil
from pathlib import Path

STAGE_LOG = Path("results_check/stage_log.json")
def run_stage(name, args, timeout_min):
    Path("results_check").mkdir(parents=True, exist_ok=True)
    rec = {"stage": name, "status": "running", "elapsed_min": None}
    t0 = time.time()
    try:
        res = subprocess.run(args, timeout=timeout_min * 60)
        rec["status"] = "ok" if res.returncode == 0 else "failed"
        rec["returncode"] = res.returncode
    except subprocess.TimeoutExpired:
        rec["status"] = "timeout"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = str(e)[:200]
    rec["elapsed_min"] = round((time.time() - t0) / 60, 1)
    entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
    entries.append(rec)
    STAGE_LOG.write_text(json.dumps(entries, indent=1))
    print(f"stage {name}: {rec['status']} ({rec['elapsed_min']}min)", flush=True)
    return rec["status"]

## 4. Gate: engine + dataset tests

In [ ]:
status = run_stage("engine_tests", [sys.executable, "scripts/test_engine.py", "--quick"], 10)
if status != "ok":
    raise RuntimeError("engine tests failed -- see output above")

## 5. Position generation sanity (tiny, exercises the oracle path)

In [ ]:
status = run_stage("gen_check", [sys.executable, "scripts/generate_positions.py", "--check", "--out", "results_check/positions"], 10)
if status != "ok":
    raise RuntimeError("position generation check failed")

## 6. The chess sweep (models x tasks, paired win/lose)

In [ ]:
status = run_stage(
    "chess_sweep",
    [sys.executable, "scripts/run_suite.py", "--output_dir", "results_check/chess" --check],
    25,
)
print("sweep:", status)

## 7. Results table

In [ ]:
import pandas as pd
csv_path = Path("results_check/chess/comparison_table.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
    print("rows:", len(df))
else:
    print("no comparison table -- sweep did not complete")

## 8. Verdict (check mode: fail loudly)

In [ ]:
entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
fails = [e for e in entries if e["status"] != "ok"]
if fails:
    raise RuntimeError(f"check mode: {len(fails)} failed stages: {[e['stage'] for e in fails]}")
print("ALL CHECK STAGES PASSED")

## Notes
- **Resume after a died session:** re-run the notebook with a trimmed sweep, e.g. `run_suite.py --models <remaining> --tasks <remaining> --output_dir results/chess`; per-run JSONs under `results/chess/*.summary.json` are the source of truth; the CSV is rebuilt at the end.
- **Gemma models** need the `HF_TOKEN` Kaggle secret (gated access).
- **Timeouts:** full-mode sweep is capped at 12h; typical T4 estimate ~1-2 min/position-cell, well under a single Kaggle session.